# Figure 1 (simplified) — single goal + single label, current `display_twist`

Reviewer 1 found the published Figure 1 (3 goals × 4 label panels) hard to read,
and the revision needs to recover a page. This notebook draws a **minimal**
version: one goal-conditioned policy panel (goal 45) beside the single
`what's labelled N` panel — the example the caption already narrates (repeating
label N traces the wall-following habit cycle into the doorway at goal 45).

It is a **separate notebook** so the original `figure-1-2.ipynb` is untouched,
and it uses the **current** `utility.display_twist` API
(`_plot_repeat_label_panel`, which reads the inverse `env.sigma_inv[s, label]`
per the corrected σ-direction convention) rather than the old `_plot_per_label`.
Output `figure1_ga_sigma_simple.{png,pdf}` is meant for side-by-side comparison
with the published `figure1_ga_sigma.pdf`.

Kernel: `py-3.12-grid`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

# gridvis pilot: co-located support + installed packages, no gridFour on path.
NB = Path.cwd()
if str(NB) not in sys.path:
    sys.path.insert(0, str(NB))

from figure_support import (
    RUN_DIR, BEST_SIGMA_HASH, ENV_ID, SHAPE, DETERMINISM, BETA, THETA,
    MAX_ITERATIONS, MAX_INFO_ITERATIONS, DI_CMAP, DI_VMIN, DI_VMAX, FIGURES_DIR,
)
from figure_support import build_env
from gridcore.info.decision_information import DecisionInformation
from gridcore.planning.policy import Policy
from gridcore.planning.state_distribution import UniformStateDistribution
from gridvis.display_twist import _plot_repeat_label_panel, _resolve_dual_plot_inputs
from gridvis import display as display_utils

FOUR_ROOMS_SIGMA_DIR = NB / 'four_rooms' / 'assets' / 'sigmas'
LOCAL_MULTI_SIGMA = FOUR_ROOMS_SIGMA_DIR / f'multi_goal_best_g200_{BEST_SIGMA_HASH}.npy'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SIMPLE_GOAL = 45        # the doorway hero the caption narrates
SIMPLE_LABEL = 'N'      # the label whose habit cycle routes into goal 45
LABEL_ORDER = ['N', 'E', 'S', 'W']
FIGURE_1_SIMPLE = FIGURES_DIR / 'figure1_ga_sigma_simple'
print('sigma exists:', LOCAL_MULTI_SIGMA.exists())

In [ ]:
def build_sigma_env(goal, sigma):
    """GridRoom with the twist applied for one goal.

    NOTE: use env.set_sigma(...) (not direct env.sigma[...] = ) so the cached
    inverse env.sigma_inv is refreshed -- the label panels read sigma_inv, and
    direct assignment leaves it at the constructor-time identity (the trap the
    grid_room docstring + the 2026-05-23 audit warn about).
    """
    sigma = np.asarray(sigma, dtype=int)
    env = build_env(ENV_ID, shape=SHAPE, goal=goal, determinism=DETERMINISM)
    full = np.tile(np.arange(env.nA, dtype=int), (env.nS, 1))   # identity rows
    avail = np.asarray(env.available_states, dtype=int)
    full[avail] = sigma[avail]
    env.set_sigma(full)            # installs sigma AND refreshes sigma_inv
    env.twist_dynamics()           # builds the twisted transition kernel
    env.update_dynamics_for_goals(env.goals)
    return env


def solve_goal(env):
    di = DecisionInformation(
        env, UniformStateDistribution(env), theta=float(THETA),
        max_iterations=int(MAX_ITERATIONS), max_info_iterations=int(MAX_INFO_ITERATIONS))
    policy, _, free = di.get_opt_policy_Z_free_vector(beta=float(BETA))
    info = di.get_decision_information_given_policy(policy).astype(float)
    value, _ = Policy.evaluate_policy(env, policy)
    return np.asarray(policy, float), info, np.asarray(free, float), value.astype(float)


def _make_ax(fig, x, y, w, h, fw, fh):
    return fig.add_axes([x / fw, y / fh, w / fw, h / fh])


def _plot_goal_panel(ax, e, p, i, goal, cmap, vmin, vmax):
    """DI heatmap + optimal-policy quiver for one goal."""
    display_utils.plot_quiver(e, p, ax=ax, skip_walls=True)
    im = display_utils.heatmap_imshow(e, i, ax, cmap=cmap, vmin=vmin, vmax=vmax)
    display_utils.add_rectangle_to_grid_for_goals(ax, e)
    display_utils.add_rectangle_to_grid_for_walls(ax, e)
    display_utils.add_outline_for_cells(ax, e)
    display_utils.add_label_to_plot(ax, i.reshape(e.shape), shift=0.35, size=7, env=e)
    ax.set_title(f'Goal {goal}', fontsize=9)
    return im

In [ ]:
# Solve the GA-best multi-goal twist at goal 45 (used for both panels).
sigma = np.load(LOCAL_MULTI_SIGMA)
env = build_sigma_env(SIMPLE_GOAL, sigma)
policy, info, _, _ = solve_goal(env)
print(f'goal {SIMPLE_GOAL}: mean_info = {float(np.dot(env.isd, info)):.4f}')

# Direction colours, consistent with the dual-panel convention.
*_, label_to_idx, color_ids, colors, _ = _resolve_dual_plot_inputs(
    env, policy, False, True, LABEL_ORDER)

In [ ]:
from matplotlib.patches import Rectangle

pw = 2.4            # panel side (in); paper sets display width via \\includegraphics
gx = 0.30; lm = rm = 0.10; tm = 0.40; bm = 0.12
fw = lm + 2 * pw + gx + rm
fh = tm + pw + bm

fig = plt.figure(figsize=(fw, fh))
ax_goal = _make_ax(fig, lm, bm, pw, pw, fw, fh)
ax_label = _make_ax(fig, lm + pw + gx, bm, pw, pw, fw, fh)

# left: goal-conditioned optimal policy + per-state DI heatmap
_plot_goal_panel(ax_goal, env, policy, info, SIMPLE_GOAL, DI_CMAP, DI_VMIN, DI_VMAX)

# --- trajectory traced by REPEATING label N from the top-left corner (state 0) ---
# Down column 0, east along row 3, down column 2, east into the goal. This is the
# "what a twist achieves" payoff: one repeated label routes the agent to the goal.
TRAJ_STATES = [0, 7, 14, 21, 22, 23, 30, 37, 44, 45]

# Soft check that the path IS the repeat-label-N rollout. env.T is a DataFrame
# indexed 'state,label' (state-major, label-minor), so reshape to (nS, nA, nS);
# its action axis is the LABEL axis after twist_dynamics. det<1 -> take argmax.
N_IDX = int(label_to_idx[SIMPLE_LABEL])
_T3 = env.T.values.reshape(env.nS, env.nA, env.nS)
_walk, _s, _goals = [0], 0, set(env.goals)
for _ in range(env.nS):
    if _s in _goals:
        break
    _ns = int(np.argmax(_T3[_s, N_IDX]))
    if _ns == _s:
        break
    _walk.append(_ns); _s = _ns
print('repeat-label-N rollout from 0:', _walk)
if _walk != TRAJ_STATES:
    print(f'  NOTE: rollout differs from spec {TRAJ_STATES}; drawing the spec path.')

# Highlight cells BEFORE the quiver so the arrows render on top. Same yellow hue
# as the left-panel goal highlight: pale yellow for the route, darker gold for goal.
_w = env.shape[1]
TRAJ_FC = (1.0, 1.0, 0.55, 0.70)   # pale light yellow (route)
GOAL_FC = (1.0, 0.78, 0.0, 0.88)   # darker gold, same hue (goal endpoint)
for _st in TRAJ_STATES:
    _gy, _gx = divmod(_st, _w)
    _fc = GOAL_FC if _st in _goals else TRAJ_FC
    ax_label.add_patch(Rectangle((_gx + 0.025, _gy + 0.025), 0.95, 0.95,
                                 linewidth=0, facecolor=_fc, zorder=0.5))

# right: 'what's labelled N' via the current API (reads env.sigma_inv[s, label]).
# exclude_goal_states=False -> show the FULL twist, including the label arrow at
# the goal cell (the label map is a property of sigma alone, not of the goal).
_plot_repeat_label_panel(
    ax_label, env, policy, int(label_to_idx[SIMPLE_LABEL]), colors,
    scale_with_prob=False, exclude_goal_states=False, show_title=False, label_states=True)
ax_label.set_title(f"what's labelled {SIMPLE_LABEL}", fontsize=9)

for ext in ('png', 'pdf'):
    fig.savefig(f'{FIGURE_1_SIMPLE}.{ext}', dpi=150, bbox_inches='tight', pad_inches=0.05)
print(f'saved -> {FIGURE_1_SIMPLE}.[png|pdf]')
plt.show()

## Candidate σ comparison — pick the most expressive labelling

Reviewer 1 found the published Figure 1 hard to read; the revision also needs to
recover a page. Beyond simplifying the *layout*, we can choose a **more
expressive twist**: one where a single label resolves to genuinely different
physical directions across the four rooms (the morphological-computation point),
rather than collapsing to "one direction everywhere".

For each candidate best-σ (the GA-selected best of a **perm_balanced** four_rooms
7×7 β=1 run, all post-submission, excluding the prod-survey class and the
fepm-bug `expl-l*` runs) this renders the goal-45 DI/policy panel **plus all four
`what's labelled N/E/S/W` panels**, saved with σ-distinguishable filenames in
`review/fig1_sigma_compare/` so the most expressive (σ, label) pair can be
chosen by eye. `current_24f55116` is the published σ for reference.

χ_twist (run-level, departure from globally-consistent labelling) and run-level
mean_info for the candidates:

| tag | χ_twist | mean_info | source run |
|---|---|---|---|
| current_24f55116 | 0.412 | 3.000 | prod-candidate (survey class) |
| **cand_38dc1f6c** | **0.425** | 2.998 | 20-03 DI, sp3011-05 |
| cand_9fe9f81d | 0.412 | 3.088 | 18-03 r3, sp3011-14 |
| cand_a3012be3 | 0.381 | 3.085 | 20-03 DI r2, sp3011-06 |


In [ ]:
# Goal-AGNOSTIC "what's labelled X" comparison across candidate sigmas.
# The label panel depends only on env.sigma_inv (scale_with_prob=False, and we
# set exclude_goal_states=False) -- it is NOT tied to any goal. So we render all
# four labels for each candidate sigma and let the most expressive (sigma, label)
# pair be chosen by eye + a heterogeneity score.

# Cohort (det= level) for the supplementary four_rooms panel: DATA_ROOT-relative
# via gridbench.store. RUN_DIR.parent is exactly the run_name-holding cohort dir.
SCHEMA10 = RUN_DIR.parent

def _run_sigma(run_name):
    return SCHEMA10 / f'run_name={run_name}' / f'{run_name}-multi-all.sigma.npy'

CANDIDATES = {
    'current_24f55116': LOCAL_MULTI_SIGMA,
    'cand_38dc1f6c': _run_sigma('g200-pop-96-perm-bal-20-03-b1-di-ga-four-rooms-sp3011-05'),
    'cand_9fe9f81d': _run_sigma('g200-pop-96-perm-bal-18-03-b1-ga-four-rooms-r3-sp3011-14'),
    'cand_a3012be3': _run_sigma('g200-pop-96-perm-bal-20-03-b1-di-ga-four-rooms-r2-sp3011-06'),
}

CMP_DIR = NB / 'review' / 'fig1_sigma_compare'
CMP_DIR.mkdir(parents=True, exist_ok=True)


def label_expressiveness(env, label_idx):
    """Heterogeneity of label->physical-direction over available (non-wall) states.

    expr = fraction of states NOT mapping to the label's modal physical direction;
    0.0 => label means one direction everywhere (un-expressive), higher => more
    spatial relabelling. Also report distinct directions used and entropy (bits).
    """
    avail = np.asarray(env.available_states, dtype=int)
    dirs = env.sigma_inv[avail, label_idx].astype(int)
    counts = np.bincount(dirs, minlength=env.nA).astype(float)
    modal_frac = counts.max() / counts.sum()
    p = counts[counts > 0] / counts.sum()
    ent = float(-(p * np.log2(p)).sum())
    return dict(expr=float(1.0 - modal_frac), n_dir=int((counts > 0).sum()), ent=ent)


rows = []
for tag, path in CANDIDATES.items():
    sigma = np.load(path)
    env = build_sigma_env(SIMPLE_GOAL, sigma)   # goal irrelevant to the label panels
    *_, label_to_idx, _, colors, _ = _resolve_dual_plot_inputs(
        env, np.full((env.nS, env.nA), 1.0 / env.nA), False, True, LABEL_ORDER)

    fig, axes = plt.subplots(1, 4, figsize=(9.6, 2.6))
    for ax, L in zip(axes, LABEL_ORDER):
        li = int(label_to_idx[L])
        m = label_expressiveness(env, li)
        rows.append((tag, L, round(m['expr'], 3), m['n_dir'], round(m['ent'], 3)))
        _plot_repeat_label_panel(
            ax, env, env.isd, li, colors,                 # policy arg unused (scale_with_prob=False)
            scale_with_prob=False, exclude_goal_states=False,  # <-- goal-agnostic
            show_title=False, label_states=True, label_size="5")
        ax.set_title(f"labelled {L}   expr={m['expr']:.2f}  ndir={m['n_dir']}", fontsize=8)
    fig.suptitle(f"{tag} — goal-agnostic 'what's labelled X'", fontsize=10, y=1.02)
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(CMP_DIR / f'whats_labelled__{tag}.{ext}', dpi=150,
                    bbox_inches='tight', pad_inches=0.05)
    plt.close(fig)
    print(f'saved whats_labelled__{tag}.[png|pdf]')

print('\n--- (sigma, label) ranked by expressiveness (frac of states off modal dir) ---')
print(f"{'sigma':<18}{'lbl':<5}{'expr':>6}{'ndir':>6}{'ent':>7}")
for tag, L, expr, nd, ent in sorted(rows, key=lambda r: -r[2]):
    print(f"{tag:<18}{L:<5}{expr:>6}{nd:>6}{ent:>7}")
print('\nfiles in', CMP_DIR)
